<a id="introduction"></a>
### Le mécanisme d'import de Python

Ce notebook montre un exemple d'application du cours.

Il reprend les différentes étapes de l'import d'un module sur un exemple.

Sous-sections :
[Chargement du module](#chargement)&nbsp;|
[Recopie des noms dans la portée courante](#recopie)&nbsp;


Un **module** est un fichier qui contient un ensemble de définitions de fonctions (ou de variables).

Un **paquetage** est un répertoire qui contient un ou plusieurs modules.

Tout comme les fonctions et les classes, les modules et les paquetages sont des _code blocks_ qui définissent chacun un espace de nommage.

Un **espace de nommage** est un ensemble de correspondances faites entre des noms
et des objets.
Techniquement, un espace de nommage est représenté par Python par
un dictionnaire qui associe des objets à des noms.


Par exemple, le module `measures` du paquetage `similarity` du paquetage `cbp` (fichier `cbp/similarity/measures.py`) contient 

```python
import math

def euclidean_sim(a,b):
    res = 0.
    for i in range(len(a)):
        res = res + (a[i] - b[i])**2
    res = math.exp(-math.sqrt(res))
    return res
```

Nous voudrions pouvoir utiliser cette fonction dans ce notebook.

Si nous appelons la fonction directement, Python lève une erreur car il ne connaît pas le symbole `euclidean_sim` (il n'a pas fait la résolution de ce nom) :


In [1]:
euclidean_sim((3,4),(3,-5))

NameError: name 'euclidean_sim' is not defined

Comment faire pour que Python fasse la **résolution des noms**, c.-à-d., sache que le nom `euclidean_sim` correspond à la fonction 
définie dans le fichier `cbp/similarity/measures.py`) ?

Le nom `euclidean_sim` est dans l'espace de nommage du module `measures`, qui n'est pour l'instant pas visible !

Pour voir l'ensemble des noms *directement accessibles* dans la portée courante (c.-à-d., visibles et utilisables sans les préfixer), on peut utiliser la fonction `dir()` :

In [ ]:
dir()

['In',
 'Out',
 '_',
 '__',
 '___',
 '__builtin__',
 '__builtins__',
 '__doc__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 '__vsc_ipynb_file__',
 '_dh',
 '_i',
 '_i1',
 '_i2',
 '_ih',
 '_ii',
 '_iii',
 '_oh',
 'exit',
 'get_ipython',
 'open',
 'quit']

On peut importer le module `measures` avec l'instruction suivante :
```python
import cbp.similarity.measures
```

Le mécanisme d'import d'un module comprend 2 étapes :
- chargement du module
- recopie dans la portée courante de certains noms définis dans le module

<a id="chargement"></a>
### Chargement du module

Python commence par regarder s'il connaît déjà le module. 
Pour cela, il cherche dans le dictionnaire `sys.modules`, où il stocke les spécifications des modules déjà importés : 

In [2]:
import sys
sys.modules['cbp']

KeyError: 'cbp'

S'il ne le trouve pas, il le cherche dans un répertoires de la variable `sys.path`.

In [ ]:
sys.path

['/home/fadi/e/src/python/m1/corriges/seance_3_modules',
 '/e/opt/miniconda3/envs/lab/lib/python311.zip',
 '/e/opt/miniconda3/envs/lab/lib/python3.11',
 '/e/opt/miniconda3/envs/lab/lib/python3.11/lib-dynload',
 '',
 '/e/opt/miniconda3/envs/lab/lib/python3.11/site-packages',
 '/e/opt/miniconda3/envs/lab/lib/python3.11/site-packages/setuptools/_vendor']

Python cherche un fichier `cbp/similarity/measures.py` dans un de ces répertoires.
S'il le trouve, il rajoute la spécification de ce module dans `sys.modules`,
ainsi que de tous les paquetages parents.

In [4]:
import cbp.similarity.measures
print(sys.modules['cbp'])
print(sys.modules['cbp.similarity'])
print(sys.modules['cbp.similarity.measures'])

<module 'cbp' (<_frozen_importlib_external.NamespaceLoader object at 0x7f902bcd3810>)>
<module 'cbp.similarity' (<_frozen_importlib_external.NamespaceLoader object at 0x7f90442fdbd0>)>
<module 'cbp.similarity.measures' from '/home/fadi/e/src/python/m1/corriges/seance_3_modules/cbp/similarity/measures.py'>


<a id="recopie"></a>
### Recopie des noms dans la portée courante

Une fois qu'il l'a trouvé et qu'il a placé sa spécification dans le dictionnaire `sys.modules`, il place certains noms de l'espace de nommage du module dans la portée courante, pour qu'on puisse les utiliser.

Si on veut savoir ce qu'il y a dans l'espace de nommage du module `measures`, on peut utiliser la fonction prédéfinie `vars()`, ou utiliser l'attribut `__dict__` :

In [ ]:
vars(cbp.similarity.measures) is cbp.similarity.measures.__dict__

True

Dans l'espace de nommage du module `measures`, un objet de type function est associé à la clé `'euclidean_sim'` :

In [5]:
vars(cbp.similarity.measures)['euclidean_sim']

<function cbp.similarity.measures.euclidean_sim(a, b)>

Mais pour l'instant, cette fonction n'est pas accessible directement.
Lors de l'import, on peut voir que seul le nom `cbp` a été placé dans la portée courante :

In [ ]:
'cbp' in dir()

True

Pour accéder au nom `euclidean_sim`, il faut le préfixer par les noms des paquetages parents :

In [ ]:
cbp.similarity.measures.euclidean_sim((3,4),(3,-5))

0.00012340980408667956

Si l'on veut pouvoir accéder directement au nom `euclidean_sim`, on peut utiliser la syntaxe suivante :

In [1]:
from cbp.similarity.measures import euclidean_sim
euclidean_sim((3,4),(3,-5))

0.00012340980408667956

Dans ce cas, le nom `euclidean_sim` est placé dans la portée courante :

In [2]:
dir()

['In',
 'Out',
 '_',
 '_1',
 '__',
 '___',
 '__builtin__',
 '__builtins__',
 '__doc__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 '__vsc_ipynb_file__',
 '_dh',
 '_i',
 '_i1',
 '_i2',
 '_ih',
 '_ii',
 '_iii',
 '_oh',
 'euclidean_sim',
 'exit',
 'get_ipython',
 'open',
 'quit']

On peut alternativement utiliser un alias pour le nom qui a été placé dans la portée courante, en utilisant le mot-clé `as` :

In [4]:
from cbp.similarity.measures import euclidean_sim as sim_eucl
sim_eucl((3,4),(3,-5))

0.00012340980408667956

Dans ce cas, le nom `sim_eucl` est placé dans la portée courante :

In [5]:
dir()

['In',
 'Out',
 'X',
 '_',
 '_1',
 '_2',
 '_3',
 '_4',
 '__',
 '___',
 '__builtin__',
 '__builtins__',
 '__doc__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 '__vsc_ipynb_file__',
 '_dh',
 '_i',
 '_i1',
 '_i2',
 '_i3',
 '_i4',
 '_i5',
 '_ih',
 '_ii',
 '_iii',
 '_oh',
 'ct',
 'euclidean_sim',
 'exit',
 'get_ipython',
 'open',
 'quit',
 'sim_eucl',
 'x',
 'y']